In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MedicalInsuranceClassification") \
    .getOrCreate()

df = spark.read.csv("/home/asih2/medical_insurance.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()


25/12/08 13:41:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+---+------+------+--------+------+---------+-----------+
|age|   sex|   bmi|children|smoker|   region|    charges|
+---+------+------+--------+------+---------+-----------+
| 19|female|  27.9|       0|   yes|southwest|  16884.924|
| 18|  male| 33.77|       1|    no|southeast|  1725.5523|
| 28|  male|  33.0|       3|    no|southeast|   4449.462|
| 33|  male|22.705|       0|    no|northwest|21984.47061|
| 32|  male| 28.88|       0|    no|northwest|  3866.8552|
+---+------+------+--------+------+---------+-----------+
only showing top 5 rows

root
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: integer (nullable = true)
 |-- smoker: string (nullable = true)
 |-- region: string (nullable = true)
 |-- charges: double (nullable = true)



In [9]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# Encode kategori
sex_indexer = StringIndexer(inputCol="sex", outputCol="sex_idx")
smoker_indexer = StringIndexer(inputCol="smoker", outputCol="label")   # target
region_indexer = StringIndexer(inputCol="region", outputCol="region_idx")

# Fitur numerik + kategori yang sudah di-encode
feature_cols = ["age", "bmi", "children", "sex_idx", "region_idx"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

pipeline = Pipeline(stages=[
    sex_indexer,
    smoker_indexer,
    region_indexer,
    assembler
])

data = pipeline.fit(df).transform(df)

data.select("features", "label").show(5, truncate=False)


+-------------------------+-----+
|features                 |label|
+-------------------------+-----+
|[19.0,27.9,0.0,1.0,1.0]  |1.0  |
|[18.0,33.77,1.0,0.0,0.0] |0.0  |
|[28.0,33.0,3.0,0.0,0.0]  |0.0  |
|[33.0,22.705,0.0,0.0,2.0]|0.0  |
|[32.0,28.88,0.0,0.0,2.0] |0.0  |
+-------------------------+-----+
only showing top 5 rows



In [10]:
train, test = data.randomSplit([0.8, 0.2], seed=42)

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)

pred = model.transform(test)
pred.show(5)


25/12/08 13:42:33 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+---+------+------+--------+------+---------+----------+-------+-----+----------+--------------------+--------------------+--------------------+----------+
|age|   sex|   bmi|children|smoker|   region|   charges|sex_idx|label|region_idx|            features|       rawPrediction|         probability|prediction|
+---+------+------+--------+------+---------+----------+-------+-----+----------+--------------------+--------------------+--------------------+----------+
| 18|female| 21.66|       0|   yes|northeast|14283.4594|    1.0|  1.0|       3.0|[18.0,21.66,0.0,1...|[1.75262834778940...|[0.85228400686530...|       0.0|
| 18|female| 25.08|       0|    no|northeast| 2196.4732|    1.0|  0.0|       3.0|[18.0,25.08,0.0,1...|[1.72657919168278...|[0.84897433817592...|       0.0|
| 18|female|26.315|       0|    no|northeast|2198.18985|    1.0|  0.0|       3.0|[18.0,26.315,0.0,...|[1.71717255197761...|[0.84776428459619...|       0.0|
| 18|female| 27.28|       3|   yes|southeast|18223.4512|    1.0|

In [11]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    metricName="accuracy"
)

auc = evaluator_auc.evaluate(pred)
acc = evaluator_acc.evaluate(pred)

print("AUC =", auc)
print("Accuracy =", acc)


AUC = 0.559330628803245
Accuracy = 0.7712031558185405


In [14]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.01, 0.1, 1.0])
             .addGrid(lr.maxIter, [20, 50, 100])
             .build())

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_auc,
    numFolds=5
)

cv_model = cv.fit(train)
best_model = cv_model.bestModel


In [15]:
print("Best regParam:", best_model._java_obj.getRegParam())
print("Best maxIter:", best_model._java_obj.getMaxIter())

Best regParam: 1.0
Best maxIter: 20


In [16]:
pred_best = best_model.transform(test)

auc_best = evaluator_auc.evaluate(pred_best)
acc_best = evaluator_acc.evaluate(pred_best)

print("Best AUC =", auc_best)
print("Best Accuracy =", acc_best)


Best AUC = 0.5564423670517681
Best Accuracy = 0.7712031558185405
